# CMDB Column Rearrange – Automatyzacja przetwarzania danych

Skrypt automatyzuje przekształcanie pliku Excel eksportowanego z systemu CMDB (np. ServiceNow)  
do formatu wymaganego przez szablon raportowy.

## Co robi skrypt:
- Wczytuje plik Excel z danymi CMDB
- Przestawia kolumny w wymaganej kolejności
- Kopiuje kolumnę `Name` i przycina wartości do hostnameu (segment przed pierwszą kropką w FQDN)
- Zapisuje gotowy plik przez okno dialogowe GUI

## Wymagania:
```
pip install pandas openpyxl
```

In [ ]:
pip install pandas openpyxl

In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from tkinter import Tk
from tkinter.filedialog import askopenfilename, asksaveasfilename


def rearrange_columns(file_path, save_path):
    """
    Przestawia kolumny pliku CMDB do wymaganej kolejności raportowej.
    Dodatkowo kopiuje kolumnę Name i przycina wartości do hostnameu (przed pierwszą kropką).

    Args:
        file_path (str): Ścieżka do pliku źródłowego Excel.
        save_path (str): Ścieżka zapisu przekształconego pliku Excel.
    """
    wb = load_workbook(filename=file_path)
    ws = wb.active
    df = pd.read_excel(file_path)

    # Wymagana kolejność kolumn wg szablonu raportowego
    # Puste stringi "" oznaczają kolumny pomocnicze bez nagłówka
    col_order = [
        "Resource Unit", "Name", "Company", "Contract", "Profitcenter",
        "RU Variant", "Name", "", "", "", "", "", "", "",
        "Description", "Status", "Class", "Assignment group",
        "Created", "Installed", "CI Managed By"
    ]

    # Budowa nowego DataFrame z wymaganą kolejnością kolumn
    new_df = pd.DataFrame()
    for col in col_order:
        if col == "":
            new_df[col] = ""
        else:
            new_df[col] = df[col]

    # Czyszczenie istniejącego arkusza
    for row in ws.iter_rows():
        for cell in row:
            cell.value = None

    # Zapis nowego DataFrame do arkusza
    for r_idx, row in enumerate(dataframe_to_rows(new_df, index=False, header=True), 1):
        for c_idx, value in enumerate(row, 1):
            ws.cell(row=r_idx, column=c_idx, value=value)

    # Kopiowanie kolumny Name (kol. 2) do kolumny 7
    ws.cell(row=1, column=7, value=ws.cell(row=1, column=2).value)
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=2, max_col=2):
        for cell in row:
            ws.cell(row=cell.row, column=7, value=cell.value)

    # Przycinanie wartości do hostnameu – zachowanie tylko segmentu przed pierwszą kropką
    # Przykład: "server01.domain.corp" → "server01"
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=7, max_col=7):
        for cell in row:
            if cell.value and isinstance(cell.value, str):
                ws.cell(row=cell.row, column=7, value=cell.value.split('.')[0])

    wb.save(save_path)
    print(f"Plik zapisany: {save_path}")


# === URUCHOMIENIE Z GUI ===
Tk().withdraw()  # Ukrycie głównego okna tkinter

source_file_path = askopenfilename(
    title="Wybierz plik CMDB (Excel)",
    filetypes=[("Excel files", "*.xlsx *.xls")]
)

if not source_file_path:
    print("Nie wybrano pliku. Kończę.")
else:
    save_file_path = asksaveasfilename(
        title="Zapisz przekształcony plik jako",
        defaultextension=".xlsx",
        filetypes=[("Excel files", "*.xlsx *.xls")]
    )
    if not save_file_path:
        print("Nie podano ścieżki zapisu. Kończę.")
    else:
        rearrange_columns(source_file_path, save_file_path)
